# Man in the Middle with DH key exchange and XOR enc/dec

## 1. Die mathematische Grundlage: Gruppenordnung und primitive Wurzeln

Beim klassischen Diffie-Hellman-Schlüsselaustausch arbeiten wir in der multiplikativen Gruppe der ganzen Zahlen modulo $p$, geschrieben als $\mathbb{Z}_p^*$, wobei $p$ eine Primzahl ist.

* Ein Element $g$ ist ein **Generator** der Gruppe $\mathbb{Z}_p^*$, wenn seine Ordnung exakt der Gruppenordnung entspricht.


* Die Ordnung der Gruppe $\mathbb{Z}_p^*$ wird durch Eulers Totient-Funktion beschrieben und beträgt für eine Primzahl $\phi(p) = p - 1$.


* Die Ordnung eines Elements $\text{ord}(g)$ ist definiert als die kleinste Zahl $k > 0$, für die $g^k = 1$ in der Gruppe gilt.


* Ein Element $g$, das die gesamte Gruppe $\mathbb{Z}_p^*$ generiert, wird auch als **primitive Wurzel modulo $p$** bezeichnet.



## 2. Die Rolle des Satzes von Lagrange

Um zu prüfen, ob eine Zahl $g$ ein Generator ist, müssten wir theoretisch prüfen, ob erst $g^{p-1} \equiv 1 \pmod p$ ist und keine kleinere Potenz vorher schon $1$ ergibt. Hier hilft der **Satz von Lagrange**: Er besagt, dass die Ordnung $\text{ord}(g)$ jedes Elements die gesamte Gruppenordnung $|G|$ (also $p-1$) ohne Rest teilen muss.

Um zu beweisen, dass $g$ ein Generator ist, reicht es also zu zeigen, dass $g$ hoch keinen der echten Teiler von $p-1$ den Wert $1$ ergibt.

## 3. Der praktische Weg: Sichere Primzahlen (Safe Primes)

Um die Suche nach einem Generator effizient zu machen, nutzt man in der Kryptographie sogenannte **sichere Primzahlen**.

* Eine Primzahl $p$ ist eine sichere Primzahl (Safe Prime), wenn $q = \frac{p-1}{2}$ ebenfalls eine Primzahl ist.


* Daraus folgt $p = 2q + 1$, und die Ordnung der Gruppe $\mathbb{Z}_p^*$ ist damit $|\mathbb{Z}_p^*| [cite_start]= 2q$.


* Aufgrund des Satzes von Lagrange hat die Gruppe nun genau 4 Untergruppen mit den Elementordnungen $1, 2, q$ und $2q$.



Da die Teiler $1, 2, q$ und $2q$ bekannt und sehr überschaubar sind, vereinfacht sich der Test massiv.

In [21]:
import sympy
# Serach for safe prime
for i in range(0, 1000):
    p: int = sympy.randprime(1000, 100000)
    q: int = (p-1) // 2
    if sympy.isprime(q):
        break
    
print(f"Using prime p = {p} with (p-1)/2 = {q} also prime")

p = 31583

Using prime p = 67883 with (p-1)/2 = 33941 also prime


## 4. Berechnung: So findest du den Generator

Dank der Struktur der sicheren Primzahl ist der Algorithmus zur Bestimmung des Generators sehr direkt:

1. 
**Kandidaten wählen:** Wähle ein zufälliges Element $g$ aus der Menge $\mathbb{Z}_p^* \setminus \{1, p-1\}$. (Wir schließen $1$ und $p-1$ aus, da diese die offensichtlichen Ordnungen $1$ und $2$ haben).


2. 
**Prüfen:** Für jedes solche $g$ gibt es laut deinen Folien nur noch zwei Möglichkeiten: Entweder generiert es eine kleinere Untergruppe der Ordnung $q$ (genannt $\mathbb{G}_q$), oder es generiert die gesamte Gruppe $\mathbb{Z}_p^*$.


3. **Berechnen:** Berechne $g^q \pmod p$.
4. **Auswerten:**
* Wenn $g^q = 1$, dann generiert $g$ "nur" die Untergruppe $\mathbb{G}_q$. (Hinweis: Für viele kryptographische Verfahren reicht genau das sogar aus ).


* Wenn $g^q \neq 1$, dann ist $g$ ein Generator für die gesamte Gruppe $\mathbb{Z}_p^*$.

In [ ]:
from random import randrange

for _ in range(100):
    g =  randrange(0, p)
    if g == 1 | g == p-1:
        continue

    if pow(g, 2, p) != 1 and pow(g, q, p) != 1:
        print(f"Found generator g: {g}")
        break
    
assert g != -1, "Error g: -1 is not valid"
    
print(f"Found generator g: {g} for prime p: {p}")

g = 24924

Found generator g: 24924
Found generator g: 24924 for prime p: 31583


In [33]:
# ALice public and private key
a = 20
A = pow(g, a, p)

# Bob message
b = 30
B = pow(g, b, p)
m = 99

# Eve Man in the Middle
e = 40
E = pow(g, e, p)

# Bob receives only E and he thinks ist form Alice
# He creates the secret key

sk_b_e = pow(E, b, p)

# XOR enc
c = m ^ sk_b_e
print(f"Bob has encrypted message m: {m}, c: {c}")

# Bob sends c and B to Eve

# Eve can decrypts the message
sk_e_b = pow(B, e, p)
m_e = c ^ sk_e_b

print(f"Eve has now the message m: {m_e}")

# Eve Manipulates the message and sends to Alice encrypted:
m_e_m = m_e + 1

sk_e_a = pow(A, e, p)
c_e_m = m_e_m ^ sk_e_a

# Eve sends the cipher text and his E to Alice
# ALice encrypt it

sk_a_e = pow(E, a, p)
 
m_a = c_e_m ^ sk_a_e
print(f"Alice has encrypted the manipulated message m_a: {m_a}")

Bob has encrypted message m: 99, c: 31606
Eve has now the message m: 99
Alice has encrypted the manipulated message m_a: 100
